# Step 0 - Look inside the Gotham dataset archive

This notebook reads `GothamDataset2025.zip` directly from Google Drive (nothing is unzipped) and reports what is inside: files, columns, labels, time structure, protocol stacks, duplicate packets and label ambiguity.

* **Runtime:** CPU is enough (*Runtime -> Change runtime type -> CPU*). Keep GPU time for training.
* **Time:** inventory 1-3 min; the full scan of ~9 GB of CSV text roughly 30-60 min.
* **Disconnected?** Reconnect and run the cells again. Finished files are skipped; only the file that was being read is redone.
* **Results:** `MyDrive/gotham_ids/reports/explore/` - `SUMMARY.md` is the file to share.

In [1]:
#@title 1. Settings
REPO_URL = "https://github.com/madhumithakk01/ds.git"  #@param {type:"string"}
BRANCH = "main"  #@param {type:"string"}
ZIP_PATH = "/content/drive/MyDrive/Ds/GothamDataset2025.zip"  # "Ds" = shortcut to the shared folder  #@param {type:"string"}
WORK_DIR = "/content/drive/MyDrive/gotham_ids"  #@param {type:"string"}

In [2]:
#@title 2. Mount Drive, get the latest code, check the environment
import os, subprocess, sys
from google.colab import drive

drive.mount("/content/drive")

REPO_DIR = "/content/ds"
if os.path.isdir(os.path.join(REPO_DIR, ".git")):
    subprocess.run(["git", "-C", REPO_DIR, "fetch", "--depth", "1", "origin", BRANCH], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "reset", "--hard", f"origin/{BRANCH}"], check=True)
else:
    subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, REPO_URL, REPO_DIR], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", os.path.join(REPO_DIR, "requirements.txt")], check=True)

os.chdir(REPO_DIR)
if not os.path.exists(ZIP_PATH):
    folder = os.path.dirname(ZIP_PATH)
    print(f"ZIP_PATH not found: {ZIP_PATH}")
    if os.path.isdir(folder):
        print(f"Contents of {folder}:")
        for name in sorted(os.listdir(folder))[:50]:
            print("  ", name)
    print("Set ZIP_PATH in cell 1 to the zip shown above (the code will also search My Drive for it).")
os.environ["GIDS_WORK_DIR"] = WORK_DIR
os.environ["GIDS_ZIP_PATH"] = ZIP_PATH
print(subprocess.run(["git", "log", "-1", "--format=code version %h (%cd)"], capture_output=True, text=True).stdout)
!python -m gids env

Mounted at /content/drive
code version 844876b (Sat Oct 17 08:04:02 2026 +0000)

Environment
  python 3.13.16 on Linux-6.6.122+-x86_64-with-glibc2.39  (colab: True)
  CPUs: 2   RAM: 12.7 GB total, 11.5 GB available
  local disk: 87.2 GB free of 107.7 GB
  GPU: none (CPU runtime)
  code: 844876b 2026-10-17T08:04:02+00:00 Look for the dataset in the Ds shortcut folder on My Drive
  Google Drive mounted: True
  packages: numpy=2.1.3, pandas=2.2.3, pyarrow=23.0.1, sklearn=1.6.1, xgboost=3.4.1, lightgbm=4.6.0, imblearn=0.14.2, matplotlib=3.10.0, yaml=6.0.3, psutil=5.9.5
  work dir: /content/drive/MyDrive/gotham_ids (free space reported: 14.9 GB)
  dataset zip: /content/drive/MyDrive/Ds/GothamDataset2025.zip -> found, 22.19 GB


In [3]:
#@title 3. Self-test on a small synthetic archive (about 1 min)
!python -m gids selftest

self-test in /tmp/gids_selftest_n0y_czgr
  [PASS] inventory runs
  [PASS] simulated crash after 3 files
  [PASS] explore resumes and finishes
  [PASS] all files scanned
  [PASS] row count exact (nothing read twice)
  [PASS] label counts exact
  [PASS] malformed lines detected
  [PASS] time parsed
  [PASS] capture joins found
  [PASS] work split across two sessions
  [PASS] summary written
  [PASS] status command
SELF-TEST PASSED


In [4]:
#@title 4. Inventory: list the archive, read the first rows of a few CSVs, measure read speed
!python -u -m gids inventory
from IPython.display import Markdown, display
display(Markdown(open(f"{WORK_DIR}/reports/explore/INVENTORY.md").read()))

2026-10-17T08:09:54Z INFO  [explore] inventory: 189 files in archive; 78 processed CSVs (6.48 GB of text)
2026-10-17T08:09:54Z INFO  [explore] schema: delimiter=comma, label column=label
2026-10-17T08:09:54Z INFO  [explore] throughput: 70.5 MB/s reading from the archive
2026-10-17T08:09:54Z INFO  [run] finish: status=ok


# Archive inventory

Source: `/content/drive/MyDrive/Ds/GothamDataset2025.zip` (22.2 GB), listed in 2.1 s

## Files by kind

| kind | files | gb | compressed_gb |
|---|---|---|---|
| other | 1 | 0 | 0 |
| processed_csv | 78 | 6.481 | 0.434 |
| raw_pcap | 110 | 23.42 | 21.75 |

## Folders

| kind | folder | files | gb |
|---|---|---|---|
| other | README.md | 1 | 0 |
| processed_csv | processed | 78 | 6.481 |
| raw_pcap | raw/benign | 78 | 6.652 |
| raw_pcap | raw/malicious/coap-amplificator | 1 | 0.066 |
| raw_pcap | raw/malicious/merlin | 7 | 1.965 |
| raw_pcap | raw/malicious/mirai-dos | 8 | 11.01 |
| raw_pcap | raw/malicious/mirai-infection | 8 | 3.36 |
| raw_pcap | raw/malicious/network-scanning | 8 | 0.364 |

## Read speed

256.0 MB of `processed/iotsim-ip-camera-museum-1.csv` in 3.63 s = **70.5 MB/s** (opening took 0.01 s)

## Head of `processed/iotsim-combined-cycle-8.csv` (0.0 MB)

delimiter: comma, label column: `label`, rows read: 254

| column | non_empty_pct | numeric_pct | hex_pct | multi_value_pct | distinct_in_head | examples |
|---|---|---|---|---|---|---|
| frame.time | 100 | 0 | 0 | 0 | 254 | ['Jan 14, 2025 18:45:08.697625000 GMT', 'Jan 14, 2025 18:45:08.700793000 GMT', ' |
| frame.len | 100 | 100 | 0 | 0 | 12 | ['67', '56', '62'] |
| frame.protocols | 100 | 0 | 0 | 0 | 2 | ['eth:ethertype:ip:udp:coap', 'eth:ethertype:ip:udp:coap:data-text-lines'] |
| eth.src | 100 | 0 | 0 | 0 | 2 | ['0c:03:36:87:00:00', '02:42:c5:6f:4c:00'] |
| eth.dst | 100 | 0 | 0 | 0 | 2 | ['02:42:c5:6f:4c:00', '0c:03:36:87:00:00'] |
| ip.dst | 100 | 0 | 0 | 0 | 2 | ['192.168.20.17', '192.168.4.1'] |
| ip.src | 100 | 0 | 0 | 0 | 2 | ['192.168.4.1', '192.168.20.17'] |
| ip.flags | 100 | 100 | 100 | 0 | 1 | ['0x02'] |
| ip.ttl | 100 | 100 | 0 | 0 | 2 | ['61', '64'] |
| ip.proto | 100 | 100 | 0 | 0 | 1 | ['17'] |
| ip.checksum | 100 | 100 | 100 | 0 | 253 | ['0x6144', '0x52e3', '0xe0c9'] |
| ip.tos | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.srcport | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.dstport | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.flags | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.window_size_value | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.window_size_scalefactor | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.checksum | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.options | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.pdu.size | 0 | 0 | 0 | 0 | 0 | [] |
| udp.srcport | 100 | 100 | 0 | 0 | 128 | ['55295', '5683', '45341'] |
| udp.dstport | 100 | 100 | 0 | 0 | 128 | ['5683', '55295', '45341'] |
| label | 100 | 0 | 0 | 0 | 1 | ['Benign'] |


time: {'column': 'frame_time', 'method': 'fmt:%b %d, %Y %H:%M:%S.%f', 'unparsed': 0, 'first': 1736880308.6976252, 'last': 1736887456.9150872}

labels in head: {'Benign': 254}

first lines:
```
"Jan 14, 2025 18:45:08.697625000 GMT",67,eth:ethertype:ip:udp:coap,0c:03:36:87:00:00,02:42:c5:6f:4c:00,192.168.20.17,192.168.4.1,0x02,61,17,0x6144,,,,,,,,,,55295,5683,Benign
"Jan 14, 2025 18:45:08.700793000 GMT",56,eth:ethertype:ip:udp:coap:data-text-lines,02:42:c5:6f:4c:00,0c:03:36:87:00:00,192.168.4.1,192.168.20.17,0x02,64,17,0x52e3,,,,,,,,,,5683,55295,Benign
"Jan 14, 2025 18:45:09.636051000 GMT",62,eth:ethertype:ip:udp:coap,0c:03:36:87:00:00,02:42:c5:6f:4c:00,192.168.20.17,192.168.4.1,0x02,61,17,0xe0c9,,,,,,,,,,45341,5683,Benign
"Jan 14, 2025 18:45:09.636230000 GMT",56,eth:ethertype:ip:udp:coap:data-text-lines,02:42:c5:6f:4c:00,0c:03:36:87:00:00,192.168.4.1,192.168.20.17,0x02,64,17,0x502b,,,,,,,,,,5683,45341,Benign
"Jan 14, 2025 18:45:10.150532000 GMT",64,eth:ethertype:ip:udp:coap,0c:03:36:87:00:00,02:42:c5:6f:4c:00,192.168.20.17,192.168.4.1,0x02,61,17,0xa78c,,,,,,,,,,36121,5683,Benign
```

## Head of `processed/iotsim-building-monitor-4.csv` (0.3 MB)

delimiter: comma, label column: `label`, rows read: 1,740

| column | non_empty_pct | numeric_pct | hex_pct | multi_value_pct | distinct_in_head | examples |
|---|---|---|---|---|---|---|
| frame.time | 100 | 0 | 0 | 0 | 1,740 | ['Jan 14, 2025 18:40:23.618578000 GMT', 'Jan 14, 2025 18:40:23.618618000 GMT', ' |
| frame.len | 100 | 100 | 0 | 0 | 23 | ['77', '93', '98'] |
| frame.protocols | 100 | 0 | 0 | 0 | 6 | ['eth:ethertype:ip:udp:dns', 'eth:ethertype:ip:icmp:data', 'eth:ethertype:ip:udp |
| eth.src | 100 | 0 | 0 | 0 | 2 | ['02:42:c3:13:3c:00', '0c:bc:fa:c1:00:00'] |
| eth.dst | 100 | 0 | 0 | 0 | 2 | ['0c:bc:fa:c1:00:00', '02:42:c3:13:3c:00'] |
| ip.dst | 100 | 0 | 0 | 0 | 4 | ['192.168.0.2', '192.168.17.13', '192.168.1.1'] |
| ip.src | 100 | 0 | 0 | 0 | 4 | ['192.168.17.13', '192.168.0.2', '192.168.1.1'] |
| ip.flags | 100 | 100 | 100 | 0 | 2 | ['0x02', '0x00'] |
| ip.ttl | 100 | 100 | 0 | 0 | 2 | ['64', '61'] |
| ip.proto | 100 | 100 | 0 | 0 | 3 | ['17', '1', '6'] |
| ip.checksum | 100 | 100 | 100 | 0 | 1,703 | ['0xd1c1', '0xd1c0', '0x4e9a'] |
| ip.tos | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.srcport | 8.966 | 100 | 0 | 0 | 13 | ['50051.0', '1883.0', '48071.0'] |
| tcp.dstport | 8.966 | 100 | 0 | 0 | 13 | ['1883.0', '50051.0', '48071.0'] |
| tcp.flags | 8.966 | 100 | 100 | 0 | 6 | ['0x0002', '0x0012', '0x0010'] |
| tcp.window_size_value | 8.966 | 100 | 0 | 0 | 5 | ['64240.0', '65160.0', '502.0'] |
| tcp.window_size_scalefactor | 7.586 | 100 | 0 | 0 | 1 | ['128.0'] |
| tcp.checksum | 8.966 | 100 | 100 | 0 | 156 | ['0x28c1', '0x7014', '0x9b6b'] |
| tcp.options | 8.966 | 0 | 0 | 0 | 99 | ['020405b40402080af7614f470000000001030307', '020405b40402080a49ae77f6f7614f4701 |
| tcp.pdu.size | 3.448 | 100 | 0 | 0 | 18 | ['14.0', '4.0', '155.0'] |
| udp.srcport | 76.32 | 100 | 0 | 0 | 521 | ['51602.0', '53.0', '49833.0'] |
| udp.dstport | 76.32 | 100 | 0 | 0 | 521 | ['53.0', '51602.0', '49833.0'] |
| label | 100 | 0 | 0 | 0 | 1 | ['Benign'] |


time: {'column': 'frame_time', 'method': 'fmt:%b %d, %Y %H:%M:%S.%f', 'unparsed': 0, 'first': 1736880023.618578, 'last': 1736887646.839155}

labels in head: {'Benign': 1740}

first lines:
```
"Jan 14, 2025 18:40:23.618578000 GMT",77,eth:ethertype:ip:udp:dns,02:42:c3:13:3c:00,0c:bc:fa:c1:00:00,192.168.0.2,192.168.17.13,0x02,64,17,0xd1c1,,,,,,,,,,51602.0,53.0,Benign
"Jan 14, 2025 18:40:23.618618000 GMT",77,eth:ethertype:ip:udp:dns,02:42:c3:13:3c:00,0c:bc:fa:c1:00:00,192.168.0.2,192.168.17.13,0x02,64,17,0xd1c0,,,,,,,,,,51602.0,53.0,Benign
"Jan 14, 2025 18:40:23.624719000 GMT",93,eth:ethertype:ip:udp:dns,0c:bc:fa:c1:00:00,02:42:c3:13:3c:00,192.168.17.13,192.168.0.2,0x02,61,17,0x4e9a,,,,,,,,,,53.0,51602.0,Benign
"Jan 14, 2025 18:40:23.624738000 GMT",77,eth:ethertype:ip:udp:dns,0c:bc:fa:c1:00:00,02:42:c3:13:3c:00,192.168.17.13,192.168.0.2,0x02,61,17,0x4ea9,,,,,,,,,,53.0,51602.0,Benign
"Jan 14, 2025 18:40:23.624845000 GMT",98,eth:ethertype:ip:icmp:data,02:42:c3:13:3c:00,0c:bc:fa:c1:00:00,192.168.1.1,192.168.17.13,0x02,64,1,0x5923,,,,,,,,,,,,Benign
```

## Head of `processed/iotsim-hydraulic-system-10.csv` (0.5 MB)

delimiter: comma, label column: `label`, rows read: 2,506

| column | non_empty_pct | numeric_pct | hex_pct | multi_value_pct | distinct_in_head | examples |
|---|---|---|---|---|---|---|
| frame.time | 100 | 0 | 0 | 0 | 2,506 | ['Jan 14, 2025 18:40:07.507650000 GMT', 'Jan 14, 2025 18:40:07.513697000 GMT', ' |
| frame.len | 100 | 100 | 0 | 0 | 13 | ['68', '66', '67'] |
| frame.protocols | 100 | 0 | 0 | 0 | 5 | ['eth:ethertype:ip:tcp:mqtt', 'eth:ethertype:ip:tcp', 'eth:ethertype:ip:udp:dns' |
| eth.src | 100 | 0 | 0 | 0 | 2 | ['02:42:c2:7a:05:00', '0c:03:36:87:00:00'] |
| eth.dst | 100 | 0 | 0 | 0 | 2 | ['0c:03:36:87:00:00', '02:42:c2:7a:05:00'] |
| ip.dst | 100 | 0 | 0 | 0 | 4 | ['192.168.4.2', '192.168.20.39', '192.168.0.2'] |
| ip.src | 100 | 0 | 0 | 0 | 4 | ['192.168.20.39', '192.168.4.2', '192.168.0.2'] |
| ip.flags | 100 | 100 | 100 | 0 | 2 | ['0x02', '0x00'] |
| ip.ttl | 100 | 100 | 0 | 0 | 2 | ['64', '61'] |
| ip.proto | 100 | 100 | 0 | 0 | 3 | ['6', '17', '1'] |
| ip.checksum | 100 | 100 | 100 | 0 | 2,187 | ['0xa562', '0xaed5', '0xaed2'] |
| ip.tos | 0 | 0 | 0 | 0 | 0 | [] |
| tcp.srcport | 75.9 | 100 | 0 | 0 | 2 | ['59563.0', '1883.0'] |
| tcp.dstport | 75.9 | 100 | 0 | 0 | 2 | ['1883.0', '59563.0'] |
| tcp.flags | 75.9 | 100 | 100 | 0 | 2 | ['0x0018', '0x0010'] |
| tcp.window_size_value | 75.9 | 100 | 0 | 0 | 223 | ['502.0', '501.0', '546.0'] |
| tcp.window_size_scalefactor | 75.9 | 100 | 0 | 0 | 1 | ['-1.0'] |
| tcp.checksum | 75.9 | 100 | 100 | 0 | 1,878 | ['0x3409', '0x927a', '0xc26f'] |
| tcp.options | 75.9 | 0 | 0 | 0 | 1,075 | ['0101080ae70590cf8b3db880', '0101080a8b3e1a18e70590cf', '0101080ae70590d58b3e1a |
| tcp.pdu.size | 25.18 | 100 | 0 | 0 | 2 | ['2.0', '7739.0'] |
| udp.srcport | 23.14 | 100 | 0 | 0 | 279 | ['58904.0', '53.0', '59518.0'] |
| udp.dstport | 23.14 | 100 | 0 | 0 | 279 | ['53.0', '58904.0', '123.0'] |
| label | 100 | 0 | 0 | 0 | 1 | ['Benign'] |


time: {'column': 'frame_time', 'method': 'fmt:%b %d, %Y %H:%M:%S.%f', 'unparsed': 0, 'first': 1736880007.5076501, 'last': 1736887610.390864}

labels in head: {'Benign': 2506}

first lines:
```
"Jan 14, 2025 18:40:07.507650000 GMT",68,eth:ethertype:ip:tcp:mqtt,02:42:c2:7a:05:00,0c:03:36:87:00:00,192.168.4.2,192.168.20.39,0x02,64,6,0xa562,,59563.0,1883.0,0x0018,502.0,-1.0,0x3409,0101080ae70590cf8b3db880,2.0,,,Benign
"Jan 14, 2025 18:40:07.513697000 GMT",66,eth:ethertype:ip:tcp,0c:03:36:87:00:00,02:42:c2:7a:05:00,192.168.20.39,192.168.4.2,0x02,61,6,0xaed5,,1883.0,59563.0,0x0010,501.0,-1.0,0x927a,0101080a8b3e1a18e70590cf,,,,Benign
"Jan 14, 2025 18:40:07.513719000 GMT",68,eth:ethertype:ip:tcp:mqtt,0c:03:36:87:00:00,02:42:c2:7a:05:00,192.168.20.39,192.168.4.2,0x02,61,6,0xaed2,,1883.0,59563.0,0x0018,501.0,-1.0,0xc26f,0101080a8b3e1a18e70590cf,2.0,,,Benign
"Jan 14, 2025 18:40:07.513737000 GMT",66,eth:ethertype:ip:tcp,02:42:c2:7a:05:00,0c:03:36:87:00:00,192.168.4.2,192.168.20.39,0x02,64,6,0xa563,,59563.0,1883.0,0x0010,502.0,-1.0,0x9271,0101080ae70590d58b3e1a18,,,,Benign
"Jan 14, 2025 18:40:37.496955000 GMT",67,eth:ethertype:ip:udp:dns,02:42:c2:7a:05:00,0c:03:36:87:00:00,192.168.0.2,192.168.20.39,0x02,64,17,0x34eb,,,,,,,,,,58904.0,53.0,Benign
```



In [ ]:
#@title 5. Full scan of every CSV (resumable: after a disconnect just run cells 2 and 5 again)
!python -u -m gids explore

In [ ]:
#@title 6. Summary and charts
from IPython.display import Image, Markdown, display
R = f"{WORK_DIR}/reports/explore"
display(Markdown(open(f"{R}/SUMMARY.md").read()))
for name in ["labels.png", "classes.png", "device_type_by_label.png", "timeline.png"]:
    if os.path.exists(f"{R}/{name}"):
        display(Image(f"{R}/{name}"))

In [ ]:
#@title 7. Plain text of the summary (copy all of it into the chat)
print(open(f"{WORK_DIR}/reports/explore/SUMMARY.md").read())

In [ ]:
#@title 8. (optional) Download the whole reports folder as a zip
import shutil
from google.colab import files
shutil.make_archive("/content/explore_reports", "zip", f"{WORK_DIR}/reports/explore")
files.download("/content/explore_reports.zip")

### If something goes wrong
* `status` shows how far each stage got and whether recent runs finished or were killed.
* `trace --errors` prints warnings and errors of the latest run with full tracebacks.
* A killed run leaves its lock behind; it is taken over automatically after 5 minutes (immediately on the same machine). `unlock --yes` removes it by hand.
* If reading the zip from Drive keeps failing, copy it to the Colab disk once (`!cp "$ZIP_PATH" /content/`) and set `ZIP_PATH = "/content/GothamDataset2025.zip"` in cell 1.

In [ ]:
!python -m gids status
!python -m gids trace --errors --tail 30